# Fase 3 — Transição para Regime Ternário via QAT Contínuo

**Notebook:** `03_ternary_transition.ipynb`  
**Fase:** 3 de 4  
**Projeto:** Arquitetura Híbrida Multimodal em torno de bitnet.cpp  

---

## Resumo

Este notebook implementa a **Fase 3** do pipeline: a conversão do backbone textual para o regime de pesos ternários `{-1, 0, +1}` via *quantization-aware training* (QAT) contínuo, conforme Seção 5.4 da especificação técnica.

O processo substitui todas as camadas `nn.Linear` do backbone por `BitLinear`, implementando quantização ternária de pesos com estimador *straight-through* para retropropagação. O estado do otimizador é reutilizado a partir da Fase 2 para reduzir *spikes* de perda, seguindo a recomendação do estudo de *continual QAT*.

A migração para ativações 4-bit (estilo BitNet a4.8) é avaliada apenas ao final desta fase, após validação da degradação por bloco.

---

## Índice

1. [Instalação de Dependências](#1)
2. [Configuração Global](#2)
3. [Montagem do Google Drive](#3)
4. [Fundamentação Teórica](#4)
5. [Implementação do BitLinear e QAT](#5)
6. [Conversão do Backbone para Regime Ternário](#6)
7. [Configuração do QAT Scheduler](#7)
8. [Carregamento do Estado do Otimizador (Fase 2)](#8)
9. [Loop de QAT Contínuo](#9)
10. [Validação da Degradação por Bloco](#10)
11. [Avaliação: 4-bit de Ativações (Opcional)](#11)
12. [Persistência dos Artefatos](#12)
13. [Conclusões e Próximos Passos](#13)

In [ ]:
!pip install -q transformers==4.44.0 accelerate==0.33.0 datasets==2.21.0 \
    sentencepiece==0.2.0 safetensors==0.4.3 einops==0.8.0
print("Instalação concluída.")

In [ ]:
import json, logging, math, random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s — %(message)s",
)
logger = logging.getLogger("phase3")

SEED: int = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TEACHER_MODEL_ID: str   = "microsoft/bitnet-b1.58-2B-4T"
DTYPE_HIGH: torch.dtype = torch.bfloat16
LR_QAT: float          = 1e-4         # LR reduzido para QAT
N_EPOCHS_QAT: int      = 3
BATCH_SIZE: int         = 4
GRAD_CLIP: float        = 1.0
WARMUP_STEPS: int       = 50
LOG_INTERVAL: int       = 50
QAT_WARMUP_FRACTION: float = 0.2      # Fração de steps para rampa gradual de quantização
ENABLE_4BIT_ACTIVATIONS: bool = False  # Ativar somente após validação completa
MAX_PPL_DEGRADATION: float = 0.10     # Degradação máxima aceitável de perplexidade (10%)
DRIVE_PROJECT_DIR: str  = "/content/drive/MyDrive/multimodal-ternary-llm"
PHASE_NAME: str         = "phase3_ternary"

logger.info("Configuração da Fase 3 inicializada.")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    logger.warning("Ambiente não-Colab.")

CHECKPOINT_DIR = Path(DRIVE_PROJECT_DIR) / "checkpoints" / PHASE_NAME
PHASE2_CKPT    = Path(DRIVE_PROJECT_DIR) / "checkpoints" / "phase2_alignment" / "phase2_aligned.pt"
METRICS_DIR    = Path(DRIVE_PROJECT_DIR) / "metrics"
for d in (CHECKPOINT_DIR, METRICS_DIR): d.mkdir(parents=True, exist_ok=True)

p0 = METRICS_DIR / "phase0_baseline_metrics.json"
PHASE0_METRICS: Dict = json.load(open(p0)) if p0.exists() else {}
TEACHER_PERPLEXITY: float = PHASE0_METRICS.get("perplexity_wikitext2", float("inf"))
D_MODEL: int = PHASE0_METRICS.get("d_model", 2048)
logger.info("Perplexidade baseline (Fase 0): %.4f", TEACHER_PERPLEXITY)

## 4. Fundamentação Teórica

### 4.1 Quantização Ternária de Pesos

O regime ternário `{-1, 0, +1}` de pesos, introduzido no BitNet b1.58, é implementado via quantização *absmean*: o fator de escala é a média do valor absoluto dos pesos e cada peso é arredondado para o inteiro mais próximo no intervalo `[-1, +1]`. Durante o treinamento, os gradientes fluem pelos pesos em precisão plena via estimador *straight-through*.

### 4.2 Continual QAT e Retenção do Estado do Otimizador

O estudo de *continual quantization-aware pre-training* citado na especificação técnica demonstra que:

- A rota 16-bit → 1.58-bit é superior ao treinamento integral em baixa precisão.
- A retenção do estado do otimizador (momentos do AdamW) da fase anterior reduz spikes de perda.
- A introdução gradual da força de quantização (ramp) oferece benefício limitado mas pode reduzir picos pontuais.

### 4.3 Ativações 4-bit (BitNet a4.8)

A migração de ativações para 4-bit — seguindo o estilo híbrido do BitNet a4.8 — é condicionada à validação de que a degradação por bloco permanece dentro da margem aceitável. Canais de outlier são o principal obstáculo: a estratégia do BitNet a4.8 combina quantização híbrida com sparsificação para mitigar esse erro. A ativação desta funcionalidade é controlada pela constante `ENABLE_4BIT_ACTIVATIONS`.

## 5. Implementação do BitLinear e QAT

In [ ]:
class BitLinear(nn.Linear):
    """
    Ternary linear projection layer implementing BitNet b1.58 quantisation.

    Replaces nn.Linear with straight-through ternary weight quantisation
    and optional 8-bit activation quantisation. During training, gradients
    flow through full-precision weights via the straight-through estimator.

    Parameters
    ----------
    in_features : int
    out_features : int
    bias : bool
        No bias by default, consistent with BitNet specification.
    quant_acts : bool
        Enable 8-bit activation quantisation. Default True.
    quant_strength : float
        Interpolation coefficient between full-precision (0.0) and fully
        ternary (1.0) regime. Used by QATScheduler for gradual ramp.
    eps : float
        Numerical stability constant.

    References
    ----------
    - BitNet b1.58: The Era of 1-bit LLMs. Ma et al., 2024.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = False,
        quant_acts: bool = True,
        quant_strength: float = 1.0,
        eps: float = 1e-8,
    ) -> None:
        super().__init__(in_features, out_features, bias=bias)
        self.quant_acts     = quant_acts
        self.quant_strength = quant_strength
        self.eps            = eps

    def _quantise_weight(self, w: torch.Tensor) -> torch.Tensor:
        """
        Apply absmean ternary quantisation to weight tensor.

        Parameters
        ----------
        w : torch.Tensor
            Full-precision weight tensor.

        Returns
        -------
        torch.Tensor
            Ternary-quantised weight in same dtype as input.
        """
        scale = w.abs().mean().clamp(min=self.eps)
        return (w / scale).round().clamp(-1, 1).to(w.dtype)

    def _quantise_activation(self, x: torch.Tensor, bits: int = 8) -> torch.Tensor:
        """
        Apply absmax quantisation to an activation tensor.

        Parameters
        ----------
        x : torch.Tensor
            Input activation tensor.
        bits : int
            Target bit-width (8 for INT8, 4 for INT4). Default 8.

        Returns
        -------
        torch.Tensor
            Quantised and dequantised activation tensor.
        """
        q_range = 2 ** (bits - 1) - 1
        scale   = x.abs().max().clamp(min=self.eps) / q_range
        return (x / scale).round().clamp(-q_range, q_range) * scale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Compute the quantisation-aware linear projection.

        Parameters
        ----------
        x : torch.Tensor
            Input of shape (..., in_features).

        Returns
        -------
        torch.Tensor
            Output of shape (..., out_features).
        """
        act_bits = 4 if ENABLE_4BIT_ACTIVATIONS else 8

        if self.quant_acts:
            x = self._quantise_activation(x, bits=act_bits)

        # Ternary weight quantisation with straight-through estimator
        w_q = self._quantise_weight(self.weight)
        # Interpolação gradual: força de quantização controlada pelo scheduler
        w_eff = self.weight + self.quant_strength * (w_q - self.weight).detach()

        return F.linear(x, w_eff, self.bias)


def convert_to_bitlinear(
    module: nn.Module,
    skip_names: Optional[List[str]] = None,
    quant_strength: float = 1.0,
) -> nn.Module:
    """
    Recursively replace all nn.Linear layers with BitLinear in-place.

    Parameters
    ----------
    module : nn.Module
        Root module to traverse and convert.
    skip_names : list of str, optional
        Layer attribute names to skip (e.g. output heads).
    quant_strength : float, optional
        Initial quantisation strength. Default 1.0.

    Returns
    -------
    nn.Module
        Modified module (in-place).
    """
    if skip_names is None:
        skip_names = []
    for name, child in module.named_children():
        if name in skip_names:
            continue
        if isinstance(child, nn.Linear) and not isinstance(child, BitLinear):
            bl = BitLinear(
                child.in_features, child.out_features,
                bias=child.bias is not None,
                quant_strength=quant_strength,
            )
            bl.weight = child.weight
            if child.bias is not None:
                bl.bias = child.bias
            setattr(module, name, bl)
        else:
            convert_to_bitlinear(child, skip_names, quant_strength)
    return module


def set_quant_strength(module: nn.Module, strength: float) -> None:
    """
    Update the quantisation strength of all BitLinear layers in a module.

    Parameters
    ----------
    module : nn.Module
        Root module.
    strength : float
        New quantisation strength in [0.0, 1.0].
    """
    for m in module.modules():
        if isinstance(m, BitLinear):
            m.quant_strength = strength


logger.info("BitLinear e utilitários de QAT definidos.")

## 6. Conversão do Backbone para Regime Ternário

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

logger.info("Carregando backbone para conversão ternária: %s", TEACHER_MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

student = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID, torch_dtype=DTYPE_HIGH, device_map="auto", trust_remote_code=True
)

# Iniciar com força de quantização 0.0 (regime FP16) para inicialização suave
logger.info("Convertendo projeções lineares para BitLinear (força inicial = 0.0)...")
student = convert_to_bitlinear(
    student,
    skip_names=["lm_head"],  # Manter cabeça de linguagem em FP16
    quant_strength=0.0,
)

n_bitlinear = sum(1 for m in student.modules() if isinstance(m, BitLinear))
n_linear    = sum(1 for m in student.modules() if isinstance(m, nn.Linear) and not isinstance(m, BitLinear))
logger.info("Conversão concluída. BitLinear: %d | nn.Linear restantes: %d", n_bitlinear, n_linear)

## 7. Configuração do QAT Scheduler

In [ ]:
class QATScheduler:
    """
    Gradual quantisation-aware training strength scheduler.

    Linearly ramps quantisation strength from 0.0 to 1.0 over warmup_steps,
    then maintains full ternary quantisation for the remainder of training.

    Parameters
    ----------
    total_steps : int
        Total QAT training steps.
    warmup_fraction : float, optional
        Fraction of total_steps to use for ramp. Default 0.2.

    Notes
    -----
    Per the continual QAT study, gradual ramp provides limited but
    non-negligible reduction in loss spikes during precision transition.
    """

    def __init__(self, total_steps: int, warmup_fraction: float = 0.2) -> None:
        self.total_steps   = total_steps
        self.warmup_steps  = max(1, int(total_steps * warmup_fraction))
        self._step: int    = 0

    def get_strength(self) -> float:
        """Return current quantisation strength in [0.0, 1.0]."""
        if self._step >= self.warmup_steps:
            return 1.0
        return self._step / self.warmup_steps

    def step(self) -> None:
        """Advance the step counter."""
        self._step = min(self._step + 1, self.total_steps)


logger.info("QATScheduler definido.")

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# Descongelar todos os parâmetros do student para QAT
for p in student.parameters(): p.requires_grad = True

trainable = sum(p.numel() for p in student.parameters() if p.requires_grad)
logger.info("Parâmetros treináveis (QAT): %s", f"{trainable:,}")

# Grupos de parâmetros diferenciados por decaimento
no_decay_kw = ("bias", "norm", "embedding")
decay_params   = [p for n, p in student.named_parameters() if p.requires_grad and not any(k in n for k in no_decay_kw)]
nodecay_params = [p for n, p in student.named_parameters() if p.requires_grad and any(k in n for k in no_decay_kw)]

optimiser = AdamW(
    [{"params": decay_params, "weight_decay": 1e-2},
     {"params": nodecay_params, "weight_decay": 0.0}],
    lr=LR_QAT, betas=(0.9, 0.95), eps=1e-8,
)

# Carregar estado do otimizador da Fase 2 se disponível
if PHASE2_CKPT.exists():
    phase2_ckpt = torch.load(PHASE2_CKPT, map_location="cpu")
    try:
        optimiser.load_state_dict(phase2_ckpt["optimiser_state"])
        logger.info("Estado do otimizador da Fase 2 carregado com sucesso.")
    except Exception as e:
        logger.warning("Não foi possível carregar estado do otimizador: %s. Reiniciando.", e)
else:
    logger.warning("Checkpoint da Fase 2 não encontrado. Otimizador inicializado do zero.")

## 9. Loop de QAT Contínuo

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

raw_train = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:5000]")
raw_val   = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")

def tokenise(examples: Dict, max_len: int = 512) -> Dict:
    return tokenizer(examples["text"], truncation=True, max_length=max_len,
                     padding="max_length", return_tensors=None)

train_tok = raw_train.map(tokenise, batched=True, remove_columns=raw_train.column_names)
val_tok   = raw_val.map(lambda e: tokenise(e, 512), batched=True, remove_columns=raw_val.column_names)
train_tok.set_format(type="torch")
val_tok.set_format(type="torch")

train_dl = DataLoader(train_tok, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_tok, batch_size=BATCH_SIZE)
logger.info("Dados QAT prontos: %d train | %d val batches", len(train_dl), len(val_dl))

In [ ]:
total_steps = N_EPOCHS_QAT * len(train_dl)
scheduler   = SequentialLR(
    optimiser,
    [LinearLR(optimiser, 1e-3, 1.0, WARMUP_STEPS),
     CosineAnnealingLR(optimiser, T_max=max(total_steps - WARMUP_STEPS, 1), eta_min=1e-6)],
    milestones=[WARMUP_STEPS],
)
qat_sched = QATScheduler(total_steps, QAT_WARMUP_FRACTION)

phase3_metrics: List[Dict] = []

for epoch in range(N_EPOCHS_QAT):
    student.train()
    epoch_loss: float = 0.0
    n_steps: int = 0

    for step, batch in enumerate(train_dl):
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = ids.clone()
        labels[mask == 0] = -100

        # Atualizar força de quantização a cada step
        strength = qat_sched.get_strength()
        set_quant_strength(student, strength)

        outputs = student(input_ids=ids, attention_mask=mask, labels=labels)
        loss    = outputs.loss

        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
        optimiser.step()
        scheduler.step()
        qat_sched.step()

        epoch_loss += loss.item()
        n_steps    += 1

        if (step + 1) % LOG_INTERVAL == 0:
            logger.info(
                "Epoch %d | Step %d | loss=%.4f | quant_strength=%.3f | lr=%.2e",
                epoch+1, step+1, epoch_loss/n_steps, strength, scheduler.get_last_lr()[0],
            )

    # Validação ao final de cada epoch
    student.eval()
    val_loss: float = 0.0
    n_val: int = 0
    with torch.no_grad():
        for vbatch in val_dl:
            if n_val >= 100: break  # Limitar para não exceder tempo do Colab
            vids  = vbatch["input_ids"].to(DEVICE)
            vmask = vbatch["attention_mask"].to(DEVICE)
            vlabs = vids.clone(); vlabs[vmask == 0] = -100
            vout  = student(input_ids=vids, attention_mask=vmask, labels=vlabs)
            val_loss += vout.loss.item()
            n_val += 1

    val_avg  = val_loss / max(n_val, 1)
    val_ppl  = math.exp(val_avg)
    degradation = (val_ppl - TEACHER_PERPLEXITY) / max(TEACHER_PERPLEXITY, 1e-8)

    logger.info(
        "Epoch %d — val_ppl=%.4f | teacher_ppl=%.4f | degradação=%.2f%%",
        epoch+1, val_ppl, TEACHER_PERPLEXITY, 100 * degradation,
    )

    if degradation > MAX_PPL_DEGRADATION:
        logger.warning(
            "ATENÇÃO: Degradação (%.2f%%) excede limite aceitável (%.2f%%). "
            "Considerar redução do LR ou revisão da temperatura de QAT.",
            100 * degradation, 100 * MAX_PPL_DEGRADATION,
        )

    ep_metrics = {
        "epoch": epoch + 1,
        "train_avg_loss": epoch_loss / max(n_steps, 1),
        "val_perplexity": val_ppl,
        "ppl_degradation_pct": 100 * degradation,
        "final_quant_strength": strength,
    }
    phase3_metrics.append(ep_metrics)

logger.info("QAT contínuo concluído.")

## 10. Validação da Degradação por Bloco

In [ ]:
# Validação da degradação de perplexidade por grupo de blocos
# Esta análise identifica blocos sensíveis à quantização para diagnóstico

logger.info("Validação da degradação por bloco iniciada...")

all_layers = list(student.model.layers)
n_layers   = len(all_layers)
BLOCK_SIZE = max(1, n_layers // 4)  # Avaliar em grupos de N/4 blocos

block_degradations: List[Dict] = []
baseline_sample = next(iter(val_dl))
vids  = baseline_sample["input_ids"][:4].to(DEVICE)
vmask = baseline_sample["attention_mask"][:4].to(DEVICE)
vlabs = vids.clone(); vlabs[vmask == 0] = -100

for group_start in range(0, n_layers, BLOCK_SIZE):
    group_end = min(group_start + BLOCK_SIZE, n_layers)

    # Temporariamente desativar quantização nos blocos do grupo (força=0)
    for i in range(group_start, group_end):
        for m in all_layers[i].modules():
            if isinstance(m, BitLinear):
                m.quant_strength = 0.0

    student.eval()
    with torch.no_grad():
        out   = student(input_ids=vids, attention_mask=vmask, labels=vlabs)
        ppl_g = math.exp(out.loss.item())

    # Reativar quantização completa
    for i in range(group_start, group_end):
        for m in all_layers[i].modules():
            if isinstance(m, BitLinear):
                m.quant_strength = 1.0

    block_degradations.append({
        "block_range": f"{group_start}-{group_end-1}",
        "ppl_without_quant": ppl_g,
    })
    logger.info(
        "Blocos %d-%d | PPL sem quantização: %.4f",
        group_start, group_end-1, ppl_g,
    )

# Restaurar força de quantização plena
set_quant_strength(student, 1.0)
logger.info("Validação por bloco concluída. Força de quantização restaurada para 1.0.")

## 11. Avaliação: 4-bit de Ativações (Opcional)

A ativação de quantização 4-bit de ativações (estilo BitNet a4.8) é condicionada à validação da degradação por bloco. Esta célula avalia o impacto se `ENABLE_4BIT_ACTIVATIONS = True`.

In [ ]:
if ENABLE_4BIT_ACTIVATIONS:
    logger.info("Avaliando impacto de ativações 4-bit...")
    student.eval()
    set_quant_strength(student, 1.0)

    with torch.no_grad():
        out_4bit = student(input_ids=vids, attention_mask=vmask, labels=vlabs)
        ppl_4bit = math.exp(out_4bit.loss.item())

    degradation_4bit = (ppl_4bit - TEACHER_PERPLEXITY) / max(TEACHER_PERPLEXITY, 1e-8)
    logger.info(
        "PPL com 4-bit activations: %.4f | Degradação: %.2f%%",
        ppl_4bit, 100 * degradation_4bit,
    )
    if degradation_4bit > MAX_PPL_DEGRADATION:
        logger.warning(
            "Ativações 4-bit causam degradação excessiva (%.2f%%). "
            "Recomenda-se manter ativações 8-bit.",
            100 * degradation_4bit,
        )
else:
    logger.info(
        "ENABLE_4BIT_ACTIVATIONS=False. Manter ativações 8-bit conforme especificação (baseline)."
    )
    print("Ativações 4-bit: desabilitadas nesta execução.")

In [ ]:
# Salvar checkpoint do modelo ternário
torch.save({
    "model_state_dict": student.state_dict(),
    "optimiser_state_dict": optimiser.state_dict(),
    "epoch": N_EPOCHS_QAT,
    "quant_strength_final": 1.0,
    "activations_4bit": ENABLE_4BIT_ACTIVATIONS,
}, CHECKPOINT_DIR / "phase3_ternary_backbone.pt")

with open(METRICS_DIR / "phase3_metrics.json", "w") as f:
    json.dump({
        "qat_per_epoch": phase3_metrics,
        "block_degradation": block_degradations,
        "enable_4bit_acts": ENABLE_4BIT_ACTIVATIONS,
    }, f, indent=2)

logger.info("Artefatos da Fase 3 persistidos.")
print("\nFase 3 concluída. Backbone ternário salvo.")

## 13. Conclusões e Próximos Passos

A Fase 3 completou a conversão do backbone para regime ternário `{-1, 0, +1}` via QAT contínuo com ramp gradual de força de quantização e retenção do estado do otimizador da fase anterior.

**Condição de prosseguimento:** A degradação de perplexidade deve estar dentro de `MAX_PPL_DEGRADATION` (10%). Caso exceda, revisar o LR do QAT ou estender o número de épocas antes de avançar para a exportação.

### Próxima Fase

Prosseguir para `04_export_deploy.ipynb` (**Fase 4**): congelamento final de pesos, exportação para o formato bitnet.cpp e benchmarks de latência/throughput.